pip install rfdetr

In [ ]:
import cv2
import numpy as np
from PIL import Image
import supervision as sv
from rfdetr import RFDETRSegNano
from rfdetr.assets.coco_classes import COCO_CLASSES

model = RFDETRSegNano()
print("Model loaded successfully!")
model.optimize_for_inference()

img_path = "sample.png"

img = Image.open(img_path).convert("RGB")
img_np = np.array(img)

dets = model.predict(img, threshold=0.5)

labels = [
    f"{COCO_CLASSES[cid]} {conf:.2f}"
    for cid, conf in zip(dets.class_id, dets.confidence)
]

annotated = img_np.copy()
annotated = sv.MaskAnnotator().annotate(annotated, dets)
annotated = sv.BoxAnnotator().annotate(annotated, dets)
annotated = sv.LabelAnnotator().annotate(annotated, dets, labels)

print(f"Objects detected: {len(dets)}")

combined = np.hstack((img_np, annotated))

scale = 0.5
combined_bgr = cv2.cvtColor(combined, cv2.COLOR_RGB2BGR)
combined_resized = cv2.resize(
    combined_bgr,
    (int(combined_bgr.shape[1] * scale), int(combined_bgr.shape[0] * scale))
)

cv2.namedWindow("Result", cv2.WINDOW_NORMAL)
cv2.imshow("Result", combined_resized)

cv2.waitKey(0)
cv2.destroyAllWindows()